# Train Attribute Heads (Occasion/Fit/Material) + A/B Evaluation

This notebook trains the remaining 3 multi-head projections using DeepFashion attributes,
then evaluates the full system (compat + style + attribute heads) against the baseline.

**Prerequisites:** compat_head and style_head already trained (from notebook 1).

**Sections:**
1. Setup & clone repo
2. Download DeepFashion dataset
3. Parse attributes & build pairs
4. Pre-compute DINOv2 embeddings
5. Train occasion/fit/material heads
6. A/B evaluation (FITB + Compatibility AUC)

## 1. Setup

In [ ]:
!git clone https://github.com/anushreeberlia/loom.git
%cd /content/loom
!pip install -q -r train/requirements.txt
!pip install -q gdown scikit-learn

In [ ]:
# If you already have the repo cloned, just pull latest
# %cd /content/loom
# !git pull

## 2. Download DeepFashion Dataset

DeepFashion Category & Attribute Prediction: 289K images, 1000 attributes.
Downloaded from Google Drive (no password required).

In [ ]:
!pip install -q gdown

# Download DeepFashion Category & Attribute Prediction benchmark
# This is the annotation + image folder from the official Google Drive
!gdown --folder "https://drive.google.com/drive/folders/0B7EVK8r0v71pQ2FuZ0k0QnhBQnc" -O data/deepfashion/

In [ ]:
# Check what we got
!ls data/deepfashion/
!find data/deepfashion/ -name "list_attr*" -type f
!find data/deepfashion/ -name "*.jpg" -type f | wc -l

### Alternative: If Google Drive fails

If the gdown folder download fails, try downloading individual files:
1. Go to https://drive.google.com/drive/folders/0B7EVK8r0v71pQ2FuZ0k0QnhBQnc
2. Download `Anno.zip` and `Img.zip` manually
3. Upload to Colab and extract

In [ ]:
# Alternative: manual upload extraction
# !unzip -q Anno.zip -d data/deepfashion/
# !unzip -q Img.zip -d data/deepfashion/

## 3. Parse Attributes & Build Training Pairs

In [ ]:
# Parse DeepFashion annotations and build contrastive pairs
# Items sharing the same attribute = positive pair
# Items with different attributes = negative pair
!python train/data/prepare_deepfashion.py --data-dir data/deepfashion/

In [ ]:
# Verify pair files were created
!ls -la data/deepfashion/*.csv
!wc -l data/deepfashion/*_pairs.csv

## 4. Pre-compute DINOv2 Embeddings

Run all DeepFashion images through frozen DINOv2 backbone.
~289K images at batch_size=64 on T4 GPU takes ~50 minutes.

In [ ]:
!python train/precompute_backbones.py --data-dir data/deepfashion/ --batch-size 64 --no-segment

In [ ]:
# Verify embeddings file
import h5py
with h5py.File('data/deepfashion/backbone_embeddings.h5', 'r') as f:
    print(f'Embeddings shape: {f["embeddings"].shape}')
    print(f'Items: {len(f["item_ids"])}')

## 5. Train Attribute Heads

Each head trains with InfoNCE contrastive loss on attribute-based pairs.
~10 min per head on T4.

In [ ]:
# Train occasion head
!python train/train_heads.py --config train/config_deepfashion.yaml --head occasion

In [ ]:
# Train fit head
!python train/train_heads.py --config train/config_deepfashion.yaml --head fit

In [ ]:
# Train material head
!python train/train_heads.py --config train/config_deepfashion.yaml --head material

In [ ]:
# Verify all weights saved
!ls -la models/multihead/

## 6. A/B Evaluation

Compare the trained multi-head system vs raw DINOv2 cosine baseline.

Uses Polyvore test data (need to have run notebook 1 first for Polyvore embeddings).

In [ ]:
# Download Polyvore test data if not already present
import os
if not os.path.exists('data/polyvore/backbone_embeddings.h5'):
    print("Need Polyvore embeddings for A/B eval.")
    print("Run notebook 1 first, or copy backbone_embeddings.h5 from that session.")
else:
    print(f"Polyvore embeddings found: {os.path.getsize('data/polyvore/backbone_embeddings.h5') / 1e6:.1f} MB")

In [ ]:
# Run A/B evaluation
!python train/eval_ab_test.py --config train/config.yaml --data-dir data/polyvore

## 7. Download Trained Weights

Download all trained head weights to use in production.

In [ ]:
from google.colab import files

# Package all weights
!tar czf multihead_weights.tar.gz models/multihead/
!ls -la multihead_weights.tar.gz

# Download to your machine
files.download('multihead_weights.tar.gz')

## Usage

After downloading `multihead_weights.tar.gz`:

```bash
cd loom/
tar xzf ~/Downloads/multihead_weights.tar.gz
# Weights are now in models/multihead/ -- production code auto-loads them
```

The inference code in `services/multihead.py` will automatically detect and load
trained weights on next server start.